# Tree-of-Thought (ToT)

*Level 8 — Reasoning Strategies*

## Objective

Instead of one linear reasoning chain, explore several candidate next-thoughts at each step,
score each with a state evaluator, and keep only the best-scoring branches -- real backtracking
away from unpromising reasoning, not just hoping the first path taken was the right one.


In [1]:
import sys
from pathlib import Path

LEVEL_DIR = Path.cwd().parent
sys.path.insert(0, str(LEVEL_DIR))
sys.path.insert(0, str(LEVEL_DIR / "tree-of-thought"))

from reasoning_common.dataset import prepare
from reasoning_common.embed import OllamaEmbedder
from reasoning_common.llm import OllamaLLM
from reasoning_common.retrieval import DenseRetriever
from thought_generator import generate_thoughts
from state_evaluator import evaluate_state
from tree_search import tree_of_thought_search

data = prepare()
embedder = OllamaEmbedder()
llm = OllamaLLM()
retriever = DenseRetriever.from_corpus(data.corpus, embedder=embedder)
print(f"corpus: {len(data.corpus)} facts, {len(data.questions)} questions")

corpus: 282 facts, 120 questions


## The two building blocks, called directly

Before running the full search, call the thought generator and state evaluator on their own, to
see exactly what each one actually does.

In [2]:
question = "Could Casio's first invention be worn around the ankle?"
retrieved = retriever.search(question, top_k=5)
context = "\n".join(data.corpus[doc_id] for doc_id, _ in retrieved)

thoughts = generate_thoughts(question, context, partial_path=[], k=3, llm=llm)
print("Candidate first thoughts:")
for i, t in enumerate(thoughts, 1):
    print(f"  {i}. {t}")

Candidate first thoughts:
  1. The yubiwa pipe is likely to be related to Japanese culture
  2. Ankle injuries are common in sports where mobility and flexibility are required
  3. Casio's invention predates Alexander Graham Bell's phone by several decades


In [3]:
print("Scoring each candidate thought:")
for t in thoughts:
    score = evaluate_state(question, context, [t], llm=llm)
    print(f"  {score:.2f}  {t}")

Scoring each candidate thought:


  0.80  The yubiwa pipe is likely to be related to Japanese culture


  0.80  Ankle injuries are common in sports where mobility and flexibility are required


  0.80  Casio's invention predates Alexander Graham Bell's phone by several decades


## What I observed (building blocks)

Three genuinely different first thoughts (a cultural tangent, a sports-injury tangent, and an
actually relevant historical-timing angle) -- but the evaluator scored all three identically at
0.80. It did not clearly discriminate between a plausible line of reasoning and two irrelevant
ones; a real, mild sign of the same evaluator-reliability question this notebook's next section
confirms much more starkly.

## The full search, on a real case where the evaluator itself goes wrong

This question needs an actual numeric comparison (Mount Fuji's height vs. the Sea of Japan's
depth) -- exactly the kind of step a confidently-wrong evaluator score can derail.

In [4]:
fuji_question = "Would the top of Mount Fuji stick out of the Sea of Japan?"
fuji_retrieved = retriever.search(fuji_question, top_k=5)
fuji_context = "\n".join(data.corpus[doc_id] for doc_id, _ in fuji_retrieved)

print("Retrieved facts:")
for doc_id, score in fuji_retrieved:
    print(f"  {score:.3f}  {data.corpus[doc_id]}")

Retrieved facts:
  0.732  The average depth of the Sea of Japan is 5,748 feet (1,752 metres) and its maximum depth is 12,276 feet (3,742 metres) Mount Fuji is 3,776.24 metres (12,389.2 ft) tall.
  0.636  Japan is a country in East Asia.
  0.566  Mount Emei is a 70 ton mountain located in China.
  0.542  RMS Titanic was a British passenger ship.
  0.531  Nissan's headquarters are located in Yokohama, Japan.


In [5]:
result = tree_of_thought_search(fuji_question, fuji_context, llm=llm)
print("Question:", fuji_question)
print("Real answer: True (Mount Fuji is 3,776m / ~12,389ft; the Sea of Japan's maximum depth is ~12,276ft)")
print()
print("Best reasoning path found:")
for step in result["best_path"]:
    print(" -", step)
print("\nBest path score:", result["best_score"])
print("Final answer:", result["answer"])
print("LLM calls:", result["llm_calls"])
print("\nFinal reasoning text:\n", result["reasoning"])

Question: Would the top of Mount Fuji stick out of the Sea of Japan?
Real answer: True (Mount Fuji is 3,776m / ~12,389ft; the Sea of Japan's maximum depth is ~12,276ft)

Best reasoning path found:
 - The Sea of Japan's maximum depth is greater than Mount Fuji's height

Best path score: 0.8
Final answer: False
LLM calls: 5

Final reasoning text:
 Answer: No.


## What I observed (the full search) -- confirms the anticipated failure mode directly

Real answer: **True**. ToT's answer: **False** -- wrong, at 5 LLM calls (5x Chain-of-Thought's
cost for this question).

The reasoning path found was: *"The Sea of Japan's maximum depth is greater than Mount Fuji's
height"* -- **this claim is backwards**. The real retrieved numbers are Mount Fuji at 12,389.2ft
and the Sea of Japan's maximum depth at 12,276ft: the mountain is *taller* than the sea is *deep*,
not the reverse. The state evaluator scored this factually-inverted claim **0.8 out of 1.0** --
high confidence in a wrong comparison -- and the search committed to it.

`01_chain_of_thought.ipynb`'s single linear pass, on the exact same question and the exact same
retrieved evidence, did the unit conversion and the comparison explicitly and got it right. Adding
branching and an evaluator did not add robustness here -- it added a second place (the evaluator's
own judgment) where the same small model could be wrong, and this run shows it actually was,
concretely and reproducibly, not just in theory.

## Common Failure Modes -- confirmed, not just anticipated

The README's original, pre-execution version of this level anticipated that the state evaluator
would have "the same unreliability every prior level found in its own judge calls." The Mount
Fuji case above is that anticipated failure, actually observed: the search's own evaluator scored
a specific reasoning step **backwards on the actual numbers** with high confidence, and the
search followed that confident-but-wrong step to an incorrect final answer, while Chain-of-Thought
(a single pass, forced to write out the real comparison) got the arithmetic right in
`01_chain_of_thought.ipynb`. Branching and scoring did not protect against a bad judgment here --
it committed to one, confidently.